# SPPR Colab Backend\n\nЭтот ноутбук поднимает FastAPI backend поверх данных из Google Drive.

In [ ]:
from google.colab import drive\ndrive.mount('/content/drive')

In [ ]:
%cd /content\n!git clone https://github.com/Nephalem72/SPPR-colab-backend.git\n%cd /content/SPPR-colab-backend\n!pip install -r requirements.txt

In [ ]:
from pathlib import Path\n\ndata_dir = Path('/content/drive/MyDrive/SPPR/data')\nrequired = [\n    'laws.parquet',\n    'final_roles_punishments_v3.parquet',\n    'cases_with_id.parquet',\n    'role_model.pkl',\n    'vectorizer.pkl',\n    'embeddings.pkl',\n    'faiss_index.bin',\n]\n\nfor name in required:\n    path = data_dir / name\n    print(name, path.exists(), path.stat().st_size if path.exists() else 'MISSING')

In [ ]:
import os\nos.environ['SPPR_DRIVE_ROOT'] = '/content/drive/MyDrive/SPPR'\nos.environ['SPPR_DATA_DIR'] = '/content/drive/MyDrive/SPPR/data'\nos.environ['SPPR_HOST'] = '0.0.0.0'\nos.environ['SPPR_PORT'] = '8000'

# Меняй эту строку для сравнения моделей. После изменения перезапусти сервер.
os.environ['SPPR_LLM_MODEL_ID'] = 'Vikhrmodels/Vikhr-Qwen-2.5-1.5B-Instruct'
os.environ['SPPR_LLM_LOAD_IN_4BIT'] = 'true'
os.environ['SPPR_RAG_PROFILE'] = 'balanced'

In [ ]:
import subprocess
import time

server = subprocess.Popen(['python', 'app_fastapi.py'])
time.sleep(3)
print('FastAPI PID:', server.pid)

In [ ]:
!curl -s http://127.0.0.1:8000/health

In [ ]:
!curl -s 'http://127.0.0.1:8000/warmup?include_llm=true'

In [ ]:
import requests

payload = {
    'text': 'А. предложил Б. совершить хищение, разработал план и распределил роли.',
    'question': 'Оцени роль А. и укажи основания со ссылками на материалы.',
    'history': [],
    'rag_profile': 'balanced',
    'return_context': False,
}
response = requests.post('http://127.0.0.1:8000/chat', json=payload, timeout=600)
response.raise_for_status()
response.json()

## Сравнение моделей

Необязательный прогон фиксированного набора запросов. Отчет появится в `eval/results/`.

In [ ]:
!python scripts/benchmark_api.py --profile balanced

Для внешнего доступа можно отдельно поднять tunnel через ngrok или cloudflared.